# ML-04 — Search Intelligence Data Contract
This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.
> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Unit of analysis + time window

*One row = what, or which dates? State it, then verify it below*


One row is one piece of content for one client's combined GSC and GA4 data on a given date. The dates are from 1/27/2025 until 6/30/2026.

In [21]:
from typing import cast
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
load_dotenv()
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

df = cast(pd.DataFrame, ds.to_pandas())

df.head(1)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30.0,0.0,115.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [22]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

| **Feature** | **Label** | **Context** | **Excluded** |
| :---     | :----:   | ---:     | ---: |
| gsc_impressions     | declined_30d_future | client_hash_id    | sessions_paid |
| gsc_clicks     | qtr_impressions_down     | content_hash_id     | ai_chatgpt |
| gsc_sum_position     | organic_ai_future     | report_date     | a_perplexity |
| gsc_avg_position     | conversion_down_future     | client_has_ga4 | ai_other |
| ga4_pageviews     | rank_future     | client_has_gsc  | ai_meta |
| ga4_engaged_sessions     | | gsc_data_available | ai_gemini |
| ga4_total_engagement_sec     | | ga4_data_available | ai_copilot |
| scroll_events     | | | ai_claude |
| sessions_organic     | | | |
| sessions_direct     | | | |
| sessions_referral     | | | |
| sessions_social     | | | |
| sessions_ai     | | | |
| ga4_sessions     | | | |
| ga4_users     | | | |

Exclusion reasoning:

sessions_paid: Paid for traffic should be irrelevant to refresh.

ai_*: AI traffic is measured but which vendor is irrelevant.

Label columns are to be added in the future.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

#### Grain Check
Check and note how many duplicate rows there are (if any). Grain should be client_hash_id + content_hash_id + report_date.

In [23]:
dup_count = df.duplicated(['client_hash_id', 'content_hash_id', 'report_date']).sum()
print(f"Duplicate rows: {dup_count}")

Duplicate rows: 6390


#### Total Rows

In [24]:
len(df)

78835655

#### Date Window

In [25]:
df['report_date'].min(), df['report_date'].max()

(datetime.date(2025, 1, 27), datetime.date(2026, 6, 30))

#### Missing Values per Field

In [26]:
df.isnull().sum() / len(df)

report_date                 0.000000
client_hash_id              0.000000
content_hash_id             0.000000
client_has_gsc              0.000000
client_has_ga4              0.000000
gsc_data_available          0.001243
ga4_data_available          0.375913
gsc_impressions             0.001243
gsc_clicks                  0.001243
gsc_sum_position            0.001243
gsc_avg_position            0.632527
ga4_pageviews               0.375913
ga4_sessions                0.375913
ga4_users                   0.375913
ga4_engaged_sessions        0.375913
ga4_total_engagement_sec    0.375913
sessions_organic            0.375913
sessions_direct             0.375913
sessions_referral           0.375913
sessions_social             0.375913
sessions_paid               0.375913
sessions_ai                 0.375913
ai_chatgpt                  0.375913
ai_perplexity               0.375913
ai_gemini                   0.375913
ai_copilot                  0.375913
ai_claude                   0.375913
a

#### Missingness by Context

In [27]:
df.groupby('gsc_data_available')['gsc_impressions'].apply(lambda x: x.isnull().mean())

gsc_data_available
False    0.0
True     0.0
Name: gsc_impressions, dtype: float64

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

#### Unbalanced history
All clients can't be compared equally because some clients have much more data than others; they start tracking at different times.

#### GA4 Gap
37.6% of rows have no available GA4 data and only GSC data. The zeroes in the GA4 columns thus aren't real tracked zeroes, they mean no data tracked.

#### Duplicates
There's 6390 rows of duplicate data, so deduplication must be handled before any modelling.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.